In [1]:
# Reproducibility run parameters (course requirement)
SID4 = 1605
SEED = 1605

import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)

import torch
torch.manual_seed(SEED)

try:
    import transformers
    transformers.set_seed(SEED)
except Exception:
    pass

print(f"SID4={SID4}, SEED={SEED} -- random/numpy/torch/transformers RNGs seeded.")


SID4=1605, SEED=1605 -- random/numpy/torch/transformers RNGs seeded.


# Part 2: Retrieval-Augmented Generation (RAG) with LangChain

This notebook builds a RAG pipeline over Wikipedia articles for 10 well-known
movies, using **explicit LCEL components** (retriever + prompt + local LLM),
FAISS for vector storage, and `sentence-transformers/all-MiniLM-L6-v2` for
embeddings. Generation uses a **local, open-source** HuggingFace model
(`google/flan-t5-base`) — no API keys, no external LLM service.

**Sections map directly to the assignment requirements:**

1. Load documents (LangChain `WikipediaLoader`)
2. Split into chunks (`RecursiveCharacterTextSplitter`, original config: `chunk_size=500, chunk_overlap=50`)
3. Embed chunks and store in FAISS
4. Build the RAG pipeline explicitly with LCEL (no `RetrievalQA`)
5. Ask 5 questions spanning multiple movies
6. Display retrieved chunks + generated answer per question
7. Rebuild with different chunk configs for 2 questions, compare side-by-side
8. Manually score retrieval success (Retrieval Success Rate)
9. Analyze at least 2 real RAG failures using the assignment's failure taxonomy

A note on the environment: the installed `transformers` version in this
environment (5.5.3) removed the legacy `text2text-generation` pipeline
factory used by seq2seq/encoder-decoder models like `flan-t5-base` (only
causal-LM `text-generation` and a few other tasks remain registered). To
keep using `flan-t5-base` through `langchain_huggingface.HuggingFacePipeline`
exactly as intended, Section 4 defines a small drop-in replacement pipeline
object that wraps `AutoModelForSeq2SeqLM` + `AutoTokenizer.generate()` and
exposes the identical call interface
(`callable(prompts) -> [{"generated_text": ...}]`, `.task` attribute) that
transformers' removed `text2text-generation` pipeline used to expose.
`HuggingFacePipeline` only relies on that interface, so this is a faithful,
minimal-surface substitute — not a different design from what the assignment
specifies.


## Setup

In [2]:
import os, re, json, time, warnings
warnings.filterwarnings("ignore")

import pandas as pd
from IPython.display import display, Markdown

DATA_DIR = "data/wikipedia_movies"
os.makedirs(DATA_DIR, exist_ok=True)

pd.set_option("display.max_colwidth", 200)

# Wikipedia's API now requires a descriptive User-Agent (their bot policy
# tightened in 2025 — anonymous/default user agents get HTTP 403). The
# `wikipedia` package (used internally by LangChain's WikipediaLoader) ships
# a generic default User-Agent that is now rejected, so we set a descriptive
# one before doing any Wikipedia requests.
import wikipedia.wikipedia as _wiki_internal
_wiki_internal.USER_AGENT = (
    "HW2-RAG-LangChain-Assignment/1.0 "
    "(educational coursework; contact: flashapps59@gmail.com)"
)
print("Setup complete.")


Setup complete.


## 1. Load Documents with a LangChain Document Loader

We use `langchain_community.document_loaders.WikipediaLoader` — a bona fide
LangChain Document Loader — to fetch one Wikipedia article per movie. Each
fetch is cached to `data/wikipedia_movies/<slug>.txt` (+ a `.meta.json`
sidecar for metadata) so re-running the notebook doesn't re-hit the network.
`WikipediaLoader` already populates `metadata['title']` and
`metadata['source']` for each `Document`, which we verify and rely on later
for citing sources next to retrieved chunks.


In [3]:
from langchain_community.document_loaders import WikipediaLoader
from langchain_core.documents import Document

MOVIES = [
    "The Shawshank Redemption",
    "The Godfather",
    "The Dark Knight",
    "Pulp Fiction",
    "Forrest Gump",
    "Inception",
    "The Matrix",
    "Fight Club",
    "Interstellar (film)",
    "Titanic (1997 film)",
]

def slug(title: str) -> str:
    return re.sub(r"[^a-z0-9]+", "_", title.lower()).strip("_")

def load_movie_documents(movies, doc_content_chars_max=20000):
    docs = []
    for title in movies:
        s = slug(title)
        cache_path = os.path.join(DATA_DIR, f"{s}.txt")
        meta_path = os.path.join(DATA_DIR, f"{s}.meta.json")
        if os.path.exists(cache_path) and os.path.exists(meta_path):
            with open(cache_path, "r", encoding="utf-8") as f:
                text = f.read()
            with open(meta_path, "r", encoding="utf-8") as f:
                meta = json.load(f)
            docs.append(Document(page_content=text, metadata=meta))
            print(f"[cache] {title:<28s} {len(text):>6d} chars")
        else:
            loaded = WikipediaLoader(
                query=title, load_max_docs=1, doc_content_chars_max=doc_content_chars_max
            ).load()
            if not loaded:
                print(f"[WARN] no Wikipedia result for {title!r}")
                continue
            d = loaded[0]
            with open(cache_path, "w", encoding="utf-8") as f:
                f.write(d.page_content)
            with open(meta_path, "w", encoding="utf-8") as f:
                json.dump(d.metadata, f)
            docs.append(d)
            print(f"[fetch] {title:<28s} {len(d.page_content):>6d} chars")
    return docs

t0 = time.time()
documents = load_movie_documents(MOVIES)
print(f"\nLoaded {len(documents)} documents in {time.time() - t0:.1f}s")


[cache] The Shawshank Redemption      20000 chars
[cache] The Godfather                 20000 chars
[cache] The Dark Knight               20000 chars
[cache] Pulp Fiction                  20000 chars
[cache] Forrest Gump                  20000 chars
[cache] Inception                     20000 chars
[cache] The Matrix                    20000 chars
[cache] Fight Club                    20000 chars
[cache] Interstellar (film)           20000 chars
[cache] Titanic (1997 film)           20000 chars

Loaded 10 documents in 0.0s


In [4]:
# Verify metadata (title / source) LangChain's WikipediaLoader attaches to each Document
meta_df = pd.DataFrame(
    [{"title": d.metadata.get("title"), "source": d.metadata.get("source"), "chars": len(d.page_content)} for d in documents]
)
meta_df


,title,source,chars
0,The Shawshank Redemption,https://en.wikipedia.org/wiki/The_Shawshank_Redemption,20000
1,The Godfather,https://en.wikipedia.org/wiki/The_Godfather,20000
2,The Dark Knight,https://en.wikipedia.org/wiki/The_Dark_Knight,20000
3,Pulp Fiction,https://en.wikipedia.org/wiki/Pulp_Fiction,20000
4,Forrest Gump,https://en.wikipedia.org/wiki/Forrest_Gump,20000
5,Inception,https://en.wikipedia.org/wiki/Inception,20000
6,The Matrix,https://en.wikipedia.org/wiki/The_Matrix,20000
7,Fight Club,https://en.wikipedia.org/wiki/Fight_Club,20000
8,Interstellar (film),https://en.wikipedia.org/wiki/Interstellar_(film),20000
9,Titanic (1997 film),https://en.wikipedia.org/wiki/Titanic_(1997_film),20000


## 2. Split into Chunks (Original Configuration)

Original configuration per the assignment spec: `chunk_size=500`,
`chunk_overlap=50`, using `RecursiveCharacterTextSplitter`.


In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

ORIGINAL_CHUNK_SIZE = 500
ORIGINAL_CHUNK_OVERLAP = 50

original_splitter = RecursiveCharacterTextSplitter(
    chunk_size=ORIGINAL_CHUNK_SIZE, chunk_overlap=ORIGINAL_CHUNK_OVERLAP
)
original_chunks = original_splitter.split_documents(documents)
print(f"Original config ({ORIGINAL_CHUNK_SIZE}/{ORIGINAL_CHUNK_OVERLAP}): {len(original_chunks)} chunks from {len(documents)} documents")

chunks_per_movie = pd.Series([c.metadata.get("title") for c in original_chunks]).value_counts()
chunks_per_movie


Original config (500/50): 590 chunks from 10 documents


Forrest Gump                61
The Matrix                  61
Fight Club                  61
Interstellar (film)         61
The Dark Knight             60
Inception                   59
The Shawshank Redemption    57
The Godfather               57
Pulp Fiction                57
Titanic (1997 film)         56
Name: count, dtype: int64

## 3. Generate Embeddings and Build the FAISS Vector Store

Embeddings: `sentence-transformers/all-MiniLM-L6-v2` via
`langchain_huggingface.HuggingFaceEmbeddings`. Vector store: FAISS
(`langchain_community.vectorstores.FAISS`), built from the original-config
chunks.


In [6]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

t0 = time.time()
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
print(f"Embeddings model loaded in {time.time() - t0:.1f}s")

t0 = time.time()
vectorstore = FAISS.from_documents(original_chunks, embeddings)
print(f"FAISS vector store built ({vectorstore.index.ntotal} vectors) in {time.time() - t0:.1f}s")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embeddings model loaded in 2.5s


FAISS vector store built (590 vectors) in 3.0s


## 4. Build the RAG Pipeline with Explicit LCEL Components

We assemble the pipeline from explicit pieces (retriever, a formatting
function, a prompt template, the local LLM, and an output parser) connected
with LangChain Expression Language (`|`) — **not** a single pre-built chain
like `RetrievalQA`. We also keep the retriever usable on its own
(`retriever.invoke(question)`) so we can display the raw retrieved chunks
separately from the generated answer (needed for Section 6).


In [7]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from langchain_huggingface import HuggingFacePipeline
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

FLAN_MODEL_ID = "google/flan-t5-base"

class Text2TextGenerationPipelineCompat:
    '''Drop-in replacement for transformers' removed text2text-generation
    pipeline (see the environment note in the intro cell). Wraps
    AutoModelForSeq2SeqLM + AutoTokenizer.generate() directly, but exposes the
    exact call interface langchain_huggingface.HuggingFacePipeline expects:
    callable(prompts) -> [{"generated_text": ...}, ...] plus a `.task`
    attribute set to "text2text-generation".
    '''
    task = "text2text-generation"

    def __init__(self, model_id: str, max_new_tokens: int = 200):
        self.tokenizer = AutoTokenizer.from_pretrained(model_id)
        self.model = AutoModelForSeq2SeqLM.from_pretrained(model_id)
        self.model.eval()
        self.max_new_tokens = max_new_tokens

    def __call__(self, prompts, **kwargs):
        single = isinstance(prompts, str)
        if single:
            prompts = [prompts]
        max_new_tokens = kwargs.get("max_new_tokens", self.max_new_tokens)
        inputs = self.tokenizer(
            prompts, return_tensors="pt", padding=True, truncation=True, max_length=512
        )
        with torch.no_grad():
            # do_sample=False (explicit) => deterministic beam search decoding;
            # no randomness enters generation regardless of the global seed.
            out_ids = self.model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                num_beams=4,
                early_stopping=True,
                do_sample=False,
            )
        texts = self.tokenizer.batch_decode(out_ids, skip_special_tokens=True)
        results = [{"generated_text": t} for t in texts]
        return results[0] if single else results

t0 = time.time()
gen_pipe = Text2TextGenerationPipelineCompat(FLAN_MODEL_ID, max_new_tokens=200)
llm = HuggingFacePipeline(pipeline=gen_pipe, model_id=FLAN_MODEL_ID)
print(f"LLM ({FLAN_MODEL_ID}) loaded in {time.time() - t0:.1f}s")


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


LLM (google/flan-t5-base) loaded in 0.8s


**Determinism note:** generation uses beam search (`num_beams=4`, `early_stopping=True`) with `do_sample=False` set explicitly on every call to `model.generate()` (see the cell above) -- flan-t5-base's default is already greedy/beam (non-sampling) decoding, and we verified this in code rather than assuming it. Combined with `SEED`-seeded `random`/`numpy`/`torch`/`transformers` RNGs (first cell) and the model running in `eval()` mode (no dropout), the generated answers are fully deterministic across re-runs on the same hardware/library versions.

In [8]:
# --- Explicit LCEL pipeline ---
prompt = ChatPromptTemplate.from_template(
    "Answer the question using only the following context.\n\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"
)

def make_retriever(vs, k=3):
    return vs.as_retriever(search_kwargs={"k": k})

def format_docs(docs):
    return "\n\n".join(f"[{d.metadata.get('title', '?')}] {d.page_content}" for d in docs)

def make_chain(retriever, llm):
    return (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )

retriever = make_retriever(vectorstore, k=3)
chain = make_chain(retriever, llm)

# Sanity check: retriever usable standalone, and chain end-to-end
_sample_docs = retriever.invoke("Who directed The Matrix?")
print("Sample retrieval (standalone retriever.invoke):")
for i, d in enumerate(_sample_docs, 1):
    print(f"  [{i}] ({d.metadata.get('title')}) {d.page_content[:80]!r}...")
print("\nSample chain.invoke answer:", chain.invoke("Who directed The Matrix?"))


Sample retrieval (standalone retriever.invoke):
  [1] (The Matrix) 'The Matrix is a 1999 science fiction–action film written and directed by the Wac'...
  [2] (The Matrix) 'In 1994, the Wachowskis presented the script for the film Assassins to Warner Br'...
  [3] (The Matrix) 'Joel Silver soon joined the project as producer. Although the film had key suppo'...



Sample chain.invoke answer: the Wachowskis


## 5 & 6. Five Questions: Retrieved Chunks + Generated Answers (Original Config)

The 5 questions below deliberately span **5 different movies** (The Matrix,
The Dark Knight, The Godfather, Titanic, Interstellar) rather than
clustering on one film. For each question we show, in order:

1. The ranked chunks retrieved by the standalone retriever (source movie +
   text), and
2. The final answer generated by the full explicit LCEL chain.


In [9]:
QUESTIONS = [
    "Who directed The Matrix and in what year was it released?",
    "Which actor played the Joker in The Dark Knight, and what happened to him after filming?",
    "What novel was The Godfather based on and who wrote it?",
    "How much did Titanic gross worldwide and when was it released in the United States?",
    "Who directed Interstellar and which actor starred as the lead astronaut?",
]

def run_and_display(question, retriever, chain, label=""):
    docs_ret = retriever.invoke(question)
    answer = chain.invoke(question)
    header = f"### {label}Q: {question}" if label else f"### Q: {question}"
    display(Markdown(header))
    rows = [
        {"rank": i, "movie": d.metadata.get("title", "?"), "chunk_text": d.page_content}
        for i, d in enumerate(docs_ret, 1)
    ]
    display(pd.DataFrame(rows))
    display(Markdown(f"**Generated answer:** {answer}"))
    display(Markdown("---"))
    return docs_ret, answer

original_results = {}
for q in QUESTIONS:
    docs_ret, answer = run_and_display(q, retriever, chain)
    original_results[q] = {"docs": docs_ret, "answer": answer}


### Q: Who directed The Matrix and in what year was it released?

,rank,movie,chunk_text
0,1,The Matrix,"The Matrix is a 1999 science fiction–action film written and directed by the Wachowskis. The first installment in the Matrix film series, it stars Keanu Reeves, Laurence Fishburne, Carrie-Anne Mos..."
1,2,The Matrix,"The Matrix was released in theaters in the United States on March 31, 1999, to widespread critical acclaim. Critics praised its innovative visual effects, action sequences, cinematography, and ent..."
2,3,The Matrix,"The Matrix is considered to be among the greatest science fiction films of all time. In 2012, it was selected for preservation in the United States National Film Registry by the Library of Congres..."


**Generated answer:** 1999

---

### Q: Which actor played the Joker in The Dark Knight, and what happened to him after filming?

...

---

### Q: Who directed Interstellar and which actor starred as the lead astronaut?

,rank,movie,chunk_text
0,1,Interstellar (film),"Interstellar is a 2014 epic science fiction drama film directed by Christopher Nolan, who co-wrote the screenplay with his brother, Jonathan Nolan. It features an ensemble cast led by Matthew McCo..."
1,2,Interstellar (film),"if Spielberg were to step away. Christopher Nolan met with Thorne, then attached as executive producer, to discuss the use of spacetime in the story. In January 2013, Paramount and Warner Bros. an..."
2,3,Interstellar (film),"to Nolan's home, where she read the script for Interstellar. In early 2013, both actors were cast in the starring roles. Jessica Chastain was contacted while she was working on Miss Julie (2014) i..."


**Generated answer:** Christopher Nolan

---

## 7. Chunking Sensitivity: Alternate Configurations on 2 Questions

We rebuild the vector store with two different alternate chunk
configurations to see how retrieval and generation change — using **two
different alt configs for variety**, applied to different questions so we
can see both directions of the size trade-off:

- **Alt Config A — bigger chunks:** `chunk_size=1000, chunk_overlap=100`,
  applied to the **Titanic gross** question (Q4). The original-config top
  chunk for this question starts mid-sentence (`"$2.264 billion, making
  Titanic..."`), a chunk-boundary split — bigger chunks should keep the
  full sentence together.
- **Alt Config B — smaller chunks:** `chunk_size=200, chunk_overlap=20`,
  applied to the **Joker/Heath Ledger** question (Q2), which retrieved
  cleanly under the original config — smaller chunks should show whether
  shrinking chunks starts fragmenting that same fact.


In [10]:
def build_pipeline(chunk_size, chunk_overlap, embeddings, llm, k=3):
    splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    chunks = splitter.split_documents(documents)
    vs = FAISS.from_documents(chunks, embeddings)
    retr = make_retriever(vs, k=k)
    ch = make_chain(retr, llm)
    return retr, ch, len(chunks)

t0 = time.time()
retriever_A, chain_A, n_chunks_A = build_pipeline(1000, 100, embeddings, llm)
print(f"Alt Config A (1000/100): {n_chunks_A} chunks, built in {time.time() - t0:.1f}s")

t0 = time.time()
retriever_B, chain_B, n_chunks_B = build_pipeline(200, 20, embeddings, llm)
print(f"Alt Config B (200/20):  {n_chunks_B} chunks, built in {time.time() - t0:.1f}s")

print(f"\nFor comparison, original config (500/50): {len(original_chunks)} chunks")


Alt Config A (1000/100): 329 chunks, built in 2.6s


Alt Config B (200/20):  1298 chunks, built in 2.7s

For comparison, original config (500/50): 590 chunks


In [11]:
Q4 = QUESTIONS[3]  # Titanic gross question
Q2 = QUESTIONS[1]  # Joker / Heath Ledger question

display(Markdown("## Alt Config A (chunk_size=1000, overlap=100) on Q4"))
docs_A, answer_A = run_and_display(Q4, retriever_A, chain_A, label="[Alt A, 1000/100] ")

display(Markdown("## Alt Config B (chunk_size=200, overlap=20) on Q2"))
docs_B, answer_B = run_and_display(Q2, retriever_B, chain_B, label="[Alt B, 200/20] ")


## Alt Config A (chunk_size=1000, overlap=100) on Q4

### [Alt A, 1000/100] Q: How much did Titanic gross worldwide and when was it released in the United States?

,rank,movie,chunk_text
0,1,Titanic (1997 film),"With an initial worldwide gross of over $1.84 billion, becoming the highest-grossing film of 1997, Titanic was the first film to reach the billion-dollar mark, and was the highest-grossing film of..."
1,2,Titanic (1997 film),"Titanic premiered at the Tokyo International Film Festival on November 1, 1997, and was released in the United States on December 19. It was distributed by Paramount Pictures in the United States ..."
2,3,Titanic (1997 film),Cameron's inspiration came from his fascination with shipwrecks. He felt a love story interspersed with human loss would be essential to convey the emotional impact of the disaster. Production beg...


**Generated answer:** were used to make the film.

---

## Alt Config B (chunk_size=200, overlap=20) on Q2

### [Alt B, 200/20] Q: Which actor played the Joker in The Dark Knight, and what happened to him after filming?

,rank,movie,chunk_text
0,1,The Dark Knight,"Although he was a fan of Batman (1989), starring Jack Nicholson as the Joker, Goyer did not consider Nicholson's portrayal scary and wanted The Dark Knight's Joker to be an unknowable, already-formed"
1,2,The Dark Knight,"Nolan was aware that Nicholson's popular portrayal of the Joker would invite comparisons to his version, and wanted an actor who could cope with the associated scrutiny. Ledger's casting in August"
2,3,The Dark Knight,"to portray the Joker. Ledger died from an accidental prescription drug overdose in January 2008, leading to widespread interest from the press and public regarding his performance. When it was"


**Generated answer:** Ledger died

---

In [12]:
# --- Side-by-side comparison tables ---
def comparison_table(question, orig_docs, orig_answer, alt_docs, alt_answer, alt_label):
    rows = []
    for i in range(max(len(orig_docs), len(alt_docs))):
        rows.append({
            "rank": i + 1,
            "original (500/50) chunk": orig_docs[i].page_content if i < len(orig_docs) else "",
            f"{alt_label} chunk": alt_docs[i].page_content if i < len(alt_docs) else "",
        })
    df = pd.DataFrame(rows)
    display(Markdown(f"### Side-by-side: {question}"))
    display(df)
    display(Markdown(f"**Original (500/50) answer:** {orig_answer}\n\n**{alt_label} answer:** {alt_answer}"))
    display(Markdown("---"))

comparison_table(
    Q4,
    original_results[Q4]["docs"], original_results[Q4]["answer"],
    docs_A, answer_A, "Alt A (1000/100)",
)
comparison_table(
    Q2,
    original_results[Q2]["docs"], original_results[Q2]["answer"],
    docs_B, answer_B, "Alt B (200/20)",
)


### Side-by-side: How much did Titanic gross worldwide and when was it released in the United States?

,rank,original (500/50) chunk,Alt A (1000/100) chunk
0,1,"$2.264 billion, making Titanic the second film to gross more than $2 billion worldwide after Avatar; as of 2026, it is the fifth-highest-grossing film. In 2017, the Library of Congress selected it...","With an initial worldwide gross of over $1.84 billion, becoming the highest-grossing film of 1997, Titanic was the first film to reach the billion-dollar mark, and was the highest-grossing film of..."
1,2,"With an initial worldwide gross of over $1.84 billion, becoming the highest-grossing film of 1997, Titanic was the first film to reach the billion-dollar mark, and was the highest-grossing film of...","Titanic premiered at the Tokyo International Film Festival on November 1, 1997, and was released in the United States on December 19. It was distributed by Paramount Pictures in the United States ..."
2,3,"Titanic premiered at the Tokyo International Film Festival on November 1, 1997, and was released in the United States on December 19. It was distributed by Paramount Pictures in the United States ...",Cameron's inspiration came from his fascination with shipwrecks. He felt a love story interspersed with human loss would be essential to convey the emotional impact of the disaster. Production beg...


**Original (500/50) answer:** $2.264 billion

**Alt A (1000/100) answer:** were used to make the film.

---

### Side-by-side: Which actor played the Joker in The Dark Knight, and what happened to him after filming?

,rank,original (500/50) chunk,Alt B (200/20) chunk
0,1,The Dark Knight was marketed with an innovative interactive viral campaign that initially focused on countering criticism of Ledger's casting by those who believed he was a poor choice to portray ...,"Although he was a fan of Batman (1989), starring Jack Nicholson as the Joker, Goyer did not consider Nicholson's portrayal scary and wanted The Dark Knight's Joker to be an unknowable, already-formed"
1,2,"Although he was a fan of Batman (1989), starring Jack Nicholson as the Joker, Goyer did not consider Nicholson's portrayal scary and wanted The Dark Knight's Joker to be an unknowable, already-for...","Nolan was aware that Nicholson's popular portrayal of the Joker would invite comparisons to his version, and wanted an actor who could cope with the associated scrutiny. Ledger's casting in August"
2,3,"efforts are derailed by the Joker, an anarchistic mastermind who seeks to test how far Batman will go to save the city from chaos. The cast includes Christian Bale, Michael Caine, Heath Ledger, Ga...","to portray the Joker. Ledger died from an accidental prescription drug overdose in January 2008, leading to widespread interest from the press and public regarding his performance. When it was"


**Original (500/50) answer:** Ledger died from an accidental prescription drug overdose

**Alt B (200/20) answer:** Ledger died

---

### Discussion: What Changed, and Why

**Q4 (Titanic gross) — original (500/50) vs. Alt A (1000/100, bigger chunks):**
Under the original config, the top-ranked chunk for this question begins
mid-sentence — `"$2.264 billion, making Titanic the second film to gross
more than $2 billion worldwide..."` — because the 500-character cut point
fell inside the sentence *"Re-releases pushed the worldwide theatrical
total to $2.264 billion..."*, separating the number from the clause that
explains what it refers to. With 1000-character chunks, that entire
sentence (and its neighbors) survives inside one chunk, so the retrieved
context is more self-contained and no longer starts on an orphaned
fragment. However, the *answer quality did not improve* — the larger
chunks also pull in more surrounding, less-relevant material (Academy Award
history, CGI/production trivia about Baja Studios and Cameron's shipwreck
research) into the top-3 context. The small local LLM, given this noisier,
larger blob of context, latched onto phrasing from the least relevant
chunk and produced a garbled, unsupported answer rather than the correct
dollar figure. This illustrates the classic trade-off: **bigger chunks
reduce mid-sentence boundary splits but dilute relevance**, which can hurt
a small model's ability to pick out the right span.

**Q2 (Joker / Heath Ledger) — original (500/50) vs. Alt B (200/20, smaller chunks):**
The original config retrieved a chunk containing the full sentence
*"...criticism of Ledger's casting by those who believed he was a poor
choice to portray the Joker. Ledger died from an accidental prescription
drug overdose..."* in a single, top-ranked chunk, and the LLM correctly
answered with the cause of death. Shrinking chunks to 200 characters splits
this same material across multiple smaller chunks — the explicit
"Ledger...Joker" casting-criticism framing and the death sentence are no
longer co-located in one chunk the way they were before — and the top
retrieved chunks shift toward less directly relevant text (the Jack
Nicholson comparison). The final answer becomes shorter and less complete
(just "Ledger died" instead of naming the cause). This illustrates the
opposite failure mode: **smaller chunks can split a single key fact (or a
fact plus its surrounding disambiguating context) across chunk
boundaries**, giving the retriever/LLM a thinner, more fragmented signal
to work with even when *a* relevant chunk is still technically retrieved.


## 8. Retrieval Success Rate (Original Config, 5 Questions)

For each of the 5 original-config questions, we manually inspected the
source Wikipedia text to determine which passage actually contains the
correct answer, then checked whether at least one of the top-3 retrieved
chunks contains that information, and at what rank the first relevant
chunk appeared.


In [13]:
retrieval_eval = [
    {
        "question": QUESTIONS[0],
        "ground_truth_passage": "The Matrix is a 1999 science fiction-action film written and directed by the Wachowskis.",
        "top3_contains_answer": "Yes",
        "first_relevant_rank": 1,
    },
    {
        "question": QUESTIONS[1],
        "ground_truth_passage": "...criticism of Ledger's casting... to portray the Joker. Ledger died from an accidental prescription drug overdose in January 2008...",
        "top3_contains_answer": "Yes",
        "first_relevant_rank": 1,
    },
    {
        "question": QUESTIONS[2],
        "ground_truth_passage": "...directed by Francis Ford Coppola, who co-wrote the screenplay with Mario Puzo based on Puzo's best-selling 1969 novel [The Godfather].",
        "top3_contains_answer": "Yes",
        "first_relevant_rank": 1,
    },
    {
        "question": QUESTIONS[3],
        "ground_truth_passage": "Re-releases pushed the worldwide theatrical total to $2.264 billion... / released in the United States on December 19 [1997].",
        "top3_contains_answer": "Yes",
        "first_relevant_rank": 1,
    },
    {
        "question": QUESTIONS[4],
        "ground_truth_passage": "Interstellar is a 2014... film directed by Christopher Nolan... an ensemble cast led by Matthew McConaughey...",
        "top3_contains_answer": "Yes",
        "first_relevant_rank": 1,
    },
]

eval_df = pd.DataFrame(retrieval_eval)
success_count = (eval_df["top3_contains_answer"] == "Yes").sum()
retrieval_success_rate = success_count / len(eval_df)

display(eval_df)
print(f"\nRetrieval Success Rate = {success_count}/{len(eval_df)} = {retrieval_success_rate:.0%}")


,question,ground_truth_passage,top3_contains_answer,first_relevant_rank
0,Who directed The Matrix and in what year was it released?,The Matrix is a 1999 science fiction-action film written and directed by the Wachowskis.,Yes,1
1,"Which actor played the Joker in The Dark Knight, and what happened to him after filming?",...criticism of Ledger's casting... to portray the Joker. Ledger died from an accidental prescription drug overdose in January 2008...,Yes,1
2,What novel was The Godfather based on and who wrote it?,"...directed by Francis Ford Coppola, who co-wrote the screenplay with Mario Puzo based on Puzo's best-selling 1969 novel [The Godfather].",Yes,1
3,How much did Titanic gross worldwide and when was it released in the United States?,Re-releases pushed the worldwide theatrical total to $2.264 billion... / released in the United States on December 19 [1997].,Yes,1
4,Who directed Interstellar and which actor starred as the lead astronaut?,Interstellar is a 2014... film directed by Christopher Nolan... an ensemble cast led by Matthew McConaughey...,Yes,1



Retrieval Success Rate = 5/5 = 100%


**Retrieval Success Rate = 5/5 = 100%** for the original (500/50)
configuration — for every question, the chunk containing the ground-truth
answer text was retrieved and ranked **#1** of the top-3. This is not
surprising given the corpus: with only 10 documents and reasonably
compact, information-dense Wikipedia lead/infobox-style paragraphs,
semantic search with `all-MiniLM-L6-v2` embeddings reliably surfaces the
single most relevant paragraph for a focused factual question. Because
retrieval was clean across the board, the more interesting failures in
this pipeline surface **downstream of retrieval** — in how the small local
LLM uses (or misuses) correctly retrieved context — which Section 9
analyzes.


## 9. Failure Analysis (at least 2 real, observed failures)

Since retrieval succeeded cleanly on all 5 original-config questions
(Section 8), the failures documented here are **LLM-level failures**: cases
where the correct context was retrieved at rank 1, but the small local
`flan-t5-base` model still produced an incomplete or unsupported answer —
plus one **chunking-level** failure identified while building the ground
truth in Section 8. We use the assignment's failure taxonomy to categorize
each.


In [14]:
display(Markdown(f"**Q1:** {QUESTIONS[0]}"))
display(Markdown(f"**Generated answer:** {original_results[QUESTIONS[0]]['answer']}"))
display(Markdown(f"**Q3:** {QUESTIONS[2]}"))
display(Markdown(f"**Generated answer:** {original_results[QUESTIONS[2]]['answer']}"))
display(Markdown(f"**Q5:** {QUESTIONS[4]}"))
display(Markdown(f"**Generated answer:** {original_results[QUESTIONS[4]]['answer']}"))
display(Markdown(f"**Q4:** {QUESTIONS[3]}"))
display(Markdown(f"**Generated answer:** {original_results[QUESTIONS[3]]['answer']}"))
display(Markdown("**Q4 rank-1 retrieved chunk (verbatim):**"))
print(original_results[QUESTIONS[3]]["docs"][0].page_content)


**Q1:** Who directed The Matrix and in what year was it released?

**Generated answer:** 1999

**Q3:** What novel was The Godfather based on and who wrote it?

**Generated answer:** Mario Puzo

**Q5:** Who directed Interstellar and which actor starred as the lead astronaut?

**Generated answer:** Christopher Nolan

**Q4:** How much did Titanic gross worldwide and when was it released in the United States?

**Generated answer:** $2.264 billion

**Q4 rank-1 retrieved chunk (verbatim):**

$2.264 billion, making Titanic the second film to gross more than $2 billion worldwide after Avatar; as of 2026, it is the fifth-highest-grossing film. In 2017, the Library of Congress selected it for preservation in the United States National Film Registry as "culturally, historically, or aesthetically significant".


### Failure 1 — Category: *Correct context, LLM answer wrong/incomplete*

**Observed in:** Q1 ("Who directed The Matrix and in what year was it
released?"), and the same pattern recurs in Q3 ("What novel was The
Godfather based on and who wrote it?") and Q5 ("Who directed Interstellar
and which actor starred as the lead astronaut?").

- Q1's rank-1 chunk explicitly contains **both** facts: *"The Matrix is a
  1999 science fiction–action film written and directed by the
  Wachowskis."* The generated answer was simply **"1999"** — it dropped
  the director entirely.
- Q3's rank-1 chunk explicitly names both the novel and its author
  (*"...co-wrote the screenplay with Mario Puzo based on Puzo's
  best-selling 1969 novel"*), but the generated answer was just
  **"Mario Puzo"** — the novel's title was never stated.
- Q5's rank-1 chunk names both the director and lead actor (*"...directed
  by Christopher Nolan... an ensemble cast led by Matthew
  McConaughey..."*), but the generated answer was just **"Christopher
  Nolan"** — the actor was dropped.

**Root cause:** `flan-t5-base` is a small (250M parameter) instruction-tuned
model whose training skews heavily toward short, single-span extractive QA.
Given a **compound** question asking for two distinct facts, it consistently
extracts only one span (typically corresponding to the earlier or more
salient sub-question) rather than compositionally assembling both facts into
one answer. This is a generation-side limitation, not a retrieval problem —
the correct information was present and top-ranked in the context every
time. A larger instruction-tuned model (e.g. `flan-t5-large`, or a modern
chat model) would very likely handle compound questions more completely,
but this is an inherent, expected limitation of using a small local
open-source model, as called out in the assignment.

### Failure 2 — Category: *Chunk boundary split key info*

**Observed in:** Q4 ("How much did Titanic gross worldwide and when was it
released in the United States?").

The rank-1 chunk retrieved under the original 500/50 config begins
mid-sentence:

> `"$2.264 billion, making Titanic the second film to gross more than $2
> billion worldwide after Avatar; as of 2026, it is the fifth-highest-grossing
> film. In 2017, the Library of Congress selected it..."`

The clause that actually explains what `$2.264 billion` refers to —
*"Re-releases pushed the worldwide theatrical total to..."* — was cut off
by the chunk boundary and left behind in a different chunk (visible as the
tail end of the rank-2 chunk in Section 6's table). The retriever still
technically "succeeded" in the sense that the correct number appeared in
the top-3 (Section 8), but the chunk itself is not self-contained: read in
isolation, `"$2.264 billion, making Titanic..."` has an unresolved
pronoun-like reference at its start. The corpus also contains a second,
different dollar figure ($1.84 billion, the *initial* worldwide gross
before re-releases) in a neighboring chunk, and the final generated answer
(**"$2.264 billion"**) never disambiguates which figure it means, nor does
it mention the requested US release date at all — a second, compounding
instance of Failure 1's compound-question pattern layered on top of this
chunking artifact.

**Root cause:** `RecursiveCharacterTextSplitter` with `chunk_size=500`
cuts on the largest available separator (paragraph/sentence/word) that
fits under the size limit, but when a single sentence carrying a critical
fact straddles the 500-character mark, the cut still lands inside — or
immediately after — that sentence, separating a data point from the
clause that gives it meaning. Section 7's Alt Config A experiment
confirms this diagnosis directly: increasing `chunk_size` to 1000 keeps
the full sentence intact in one chunk (fixing this specific boundary
split), though at the cost of pulling in more tangential content that
introduced a *different* generation-side failure.


## Summary

- **10 movies** loaded via `WikipediaLoader` (LangChain Document Loader),
  cached to `data/wikipedia_movies/`.
- **Original config** (`chunk_size=500, chunk_overlap=50`) produced
  `590` chunks, embedded with `all-MiniLM-L6-v2`, stored in FAISS.
- RAG pipeline built with **explicit LCEL** components (retriever |
  format_docs, prompt, local `flan-t5-base` LLM, output parser) — no
  `RetrievalQA`.
- **5 questions** spanning 5 different movies were answered; retrieved
  chunks and answers displayed per question.
- **2 questions** re-run under 2 different alternate chunk configs
  (1000/100 and 200/20), compared side-by-side against the original.
- **Retrieval Success Rate = 5/5 = 100%** (all correct passages ranked #1).
- **2 categorized failures** analyzed: *correct context but LLM
  wrong/incomplete* (compound questions) and *chunk boundary split key
  info* (Titanic gross figure split from its explanatory clause).
- **No model checkpoint applies to this notebook:** nothing is trained here -- both the embedding model (`all-MiniLM-L6-v2`) and the generation model (`flan-t5-base`) are used strictly for frozen, pretrained inference (no fine-tuning, no weight updates), so there is no checkpoint to save.
